# 20 — DaeBuilder, FMI, and the plugin map

**Objectives.** Know when to reach for model-exchange tooling, discover installed plugins, and distinguish portable core examples from optional integrations.

`DaeBuilder` organizes time, states, derivatives, algebraic variables, controls, parameters, outputs, equations, and events. CasADi 3.7 can import FMI 2/3 FMUs through its native interface. FMU behavior depends on the archive and platform, so this portable course explains the workflow without committing a third-party binary.


In [1]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Inspect model-building and plugin capabilities

`ca.DaeBuilder` provides an incremental, named representation for
differential-algebraic models and imported model-exchange workflows. An
empty builder is sufficient to expose its interface here. The cell also
parses `CasadiMeta.plugins()` to group the solver and numerical backends
compiled into the active CasADi installation.

A general semi-explicit DAE represented by a populated builder has the form

$$
\dot x=f(t,x,z,u,p),\qquad
0=g(t,x,z,u,p),\qquad
y=h(t,x,z,u,p).
$$

Plugin families provide algorithms that operate on related mathematical
objects: `Integrator` advances ODEs/DAEs, `Rootfinder` solves
$g(z,p)=0$, `Nlpsol` solves nonlinear programs, `Conic` handles
quadratic/conic problems, `Linsol` solves linear systems, and `Interpolant`
constructs lookup-table functions. The printed list is capability
discovery, not a guarantee that every plugin is appropriate for every
model.


In [2]:
import casadi as ca

# DaeBuilder incrementally describes named differential-algebraic models.
# This empty instance exposes the API without assuming an FMU workflow.
builder = ca.DaeBuilder("empty_teaching_model")
print(builder)

# CasadiMeta.plugins returns semicolon-separated Family::plugin entries
# for every backend compiled into the active CasADi distribution.
installed_plugins = ca.CasadiMeta.plugins().split(";")
plugins_by_family = {}
for plugin_entry in installed_plugins:
    family, separator, plugin_name = plugin_entry.partition("::")
    if separator:
        plugins_by_family.setdefault(family, []).append(plugin_name)

plugin_families = [
    "Nlpsol",
    "Conic",
    "Integrator",
    "Rootfinder",
    "Linsol",
    "Interpolant",
]
for family in plugin_families:
    print(family, sorted(plugins_by_family.get(family, [])))

nx = 0, nz = 0, nq = 0, ny = 0, np = 0, nc = 0, nd = 0, nw = 0, nu = 0
Nlpsol ['fatrop', 'feasiblesqpmethod', 'ipopt', 'qrsqp', 'scpgen', 'sqpmethod']
Conic ['fatrop', 'ipqp', 'nlpsol', 'osqp', 'proxqp', 'qpoases', 'qrqp']
Integrator ['collocation', 'cvodes', 'idas', 'rk']
Rootfinder ['fast_newton', 'kinsol', 'newton', 'nlpsol']
Linsol ['csparse', 'csparsecholesky', 'lapacklu', 'lapackqr', 'ldl', 'lsqr', 'mumps', 'qr', 'symbolicqr', 'tridiag']
Interpolant ['bspline', 'linear']


## FMU workflow

1. Validate the FMU independently and record FMI version/platform support.
2. Construct/import a `DaeBuilder`, inspect variable categories, units, causality, equations, and events.
3. Create selected input/output/Jacobian functions with the function factory.
4. Compare nominal outputs and event behavior against the source model.
5. Check which generated functions support differentiation and code generation before embedding them in an NLP.

## Specialized functionality map

Use `nlpsol`, `conic`, `integrator`, `rootfinder`, `linsol`, and `interpolant` plugin registries rather than assuming a plugin ships in every wheel. Proprietary solvers need their own installation/licenses. Lower-level sparsity ordering/coloring, custom `Callback` derivatives, external functions, serialization, importers, and solver-specific options belong in the API reference once the core workflow is understood.

**Recap.** Discover capabilities at runtime, keep optional integrations isolated, and test imported models numerically before differentiating or optimizing them.
